In [6]:
!pip install -q -U langchain-text-splitters chromadb sentence-transformers

import os
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter

os.makedirs("docs", exist_ok=True)

article_text = """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own. This approach powers everything from spam filters
to recommendation systems.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Natural Language Processing, or NLP, is the branch of AI that focuses specifically on human
language. NLP powers chatbots, translation tools, and search engines. A key building block of
modern NLP is the embedding, which converts words or sentences into numeric vectors that capture
meaning.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.

Vector Databases store embeddings and allow fast similarity search across millions of documents.
They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation
engines used by companies around the world today.
"""

with open("docs/ai_overview.txt", "w") as f:
    f.write(article_text)

print("Document created: docs/ai_overview.txt")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(article_text)

print("\nNumber of chunks created:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ({len(chunk)} characters) ---")
    print(chunk)

chunk_metadatas = [
    {
        "source": "ai_overview.txt",
        "chunk_index": i,
        "char_count": len(chunk)
    }
    for i, chunk in enumerate(chunks)
]

chunk_ids = [
    f"ai_overview_chunk_{i}"
    for i in range(len(chunks))
]

print("\nExample metadata:")
print(chunk_metadatas[0])

print("\nExample ID:")
print(chunk_ids[0])

client = chromadb.Client()

try:
    client.delete_collection("rag_chunks")
except Exception:
    pass

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
    name="rag_chunks",
    embedding_function=hf_embedder
)

print("\nCollection created:", collection.name)

collection.add(
    documents=chunks,
    metadatas=chunk_metadatas,
    ids=chunk_ids
)

print("Total chunks stored:", collection.count())

stored = collection.get(
    ids=chunk_ids,
    include=["documents", "metadatas"]
)

print("\nStored Chunks:")

for cid, doc, meta in zip(
    stored["ids"],
    stored["documents"],
    stored["metadatas"]
):
    print(f"\nID: {cid}")
    print(f"Metadata: {meta}")
    print(f"Text: {doc[:100]}...")
    print("-" * 60)

results = collection.query(
    query_texts=[
        "How does RAG combine search with language models?"
    ],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)

print("\nRAG Query Results:")

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(
        f"Distance: {distance:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk #{meta['chunk_index']}"
    )
    print(doc)
    print("-" * 60)

second_doc_text = """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.
"""

with open("docs/embeddings_in_practice.txt", "w") as f:
    f.write(second_doc_text)

second_doc_chunks = splitter.split_text(second_doc_text)

second_doc_metadatas = [
    {
        "source": "embeddings_in_practice.txt",
        "chunk_index": i,
        "char_count": len(chunk)
    }
    for i, chunk in enumerate(second_doc_chunks)
]

second_doc_ids = [
    f"embeddings_in_practice_chunk_{i}"
    for i in range(len(second_doc_chunks))
]

collection.add(
    documents=second_doc_chunks,
    metadatas=second_doc_metadatas,
    ids=second_doc_ids
)

print("\nSecond document added:")
print("Chunks added:", len(second_doc_chunks))
print("Total chunks:", collection.count())

results = collection.query(
    query_texts=[
        "How do search engines use embeddings?"
    ],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print("\nSearch Across Both Documents:")

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(
        f"Distance: {distance:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk #{meta['chunk_index']}"
    )
    print(doc)
    print("-" * 60)

results = collection.query(
    query_texts=["What is deep learning?"],
    n_results=2,
    where={"source": "ai_overview.txt"},
    include=["documents", "metadatas", "distances"]
)

print("\nFiltered Search - ai_overview.txt:")

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(
        f"Distance: {distance:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk #{meta['chunk_index']}"
    )
    print(doc)
    print("-" * 60)

print("\nFinal Collection Information:")
print("Collection name:", collection.name)
print("Total chunks stored:", collection.count())

print("\nRAG Step 2 completed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 20.3 MB/s eta 0:00:00
Document created: docs/ai_overview.txt

Number of chunks created: 9

--- Chunk 0 (36 characters) ---
Artificial Intelligence: An Overview

--- Chunk 1 (270 characters) ---
Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

--- Chunk 2 (291 characters) ---
Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own. This approach powers everything from spam filters

--- Chunk 3 (26 characters) ---
to recommendation systems.

--- Chunk 4 (260 characters) ---
Deep Learning takes Machine Learning further by using ne